In [11]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("Fisrt_test") \
    .config("spark.some.config.option", "some-value") \
    .getOrCreate()

In [15]:
customers_df = spark.read.load("olist_customers_dataset.csv",
    format="csv",
    inferSchema="true",
    header="true")
customers_df.show(5)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
+--------------------+--------------------+------------------------+--------------------+--------------+
only showing top 5 rows


In [16]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [18]:
customers_df.select("customer_city").show(10)

+--------------------+
|       customer_city|
+--------------------+
|              franca|
|sao bernardo do c...|
|           sao paulo|
|     mogi das cruzes|
|            campinas|
|      jaragua do sul|
|           sao paulo|
|             timoteo|
|            curitiba|
|      belo horizonte|
+--------------------+
only showing top 10 rows


In [35]:
customers_df.filter(customers_df["customer_city"]== "curitiba").show(10)
customers_df.groupby("customer_city").count().sort("count", ascending=False).show()

+--------------------+--------------------+------------------------+-------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+--------------------+--------------------+------------------------+-------------+--------------+
|5adf08e34b2e99398...|1175e95fb47ddff9d...|                   81560|     curitiba|            PR|
|237098a64674ae89b...|4390ddbb6276a66ff...|                   82820|     curitiba|            PR|
|469634941c27cd844...|ef07ba9aa5226f772...|                   81750|     curitiba|            PR|
|aa0fbd830c89acc3d...|eabd76f3506262b0d...|                   80050|     curitiba|            PR|
|cfde763c6872e6941...|3fdc39171d444e1a1...|                   81250|     curitiba|            PR|
|8fb95d3058d1550db...|e5d46317b6efaf96e...|                   82310|     curitiba|            PR|
|f0eccf2fbfbabd057...|6995203f8a5ddbd2f...|                   81670|     curitiba|            PR|
|7eab558047c9de471..

In [ ]:
dfs = {}

datasets = {
    "customers" : "olist_customers_dataset.csv",
    "geolocation" : "olist_geolocation_dataset.csv",
    "order_items" : "olist_order_items_dataset.csv",
    "order_payments" : "olist_order_payments_dataset.csv",
    "order_reviews" : "olist_order_reviews_dataset.csv",
    "orders" : "olist_orders_dataset.csv",
    "products" : "olist_products_dataset.csv",
    "sellers" : "olist_sellers_dataset.csv",
    "product_category_translation" : "product_category_name_translation.csv"
}

for title, dataset in datasets.items():
    df = spark.read.load(dataset,
    format="csv",
    inferSchema="true",
    header="true")
    print(f"=== {title} ===")
    df.printSchema()
    print(f"Lines: {df.count()}")
    df.show(5)

    dfs[title] = df
    # df.write.mode("overwrite").parquet(f"../bronze/{title}")

=== customers ===
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

Lines: 99441
+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|   

=== geolocation ===
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

Lines: 1000163
+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|   geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|                       1037| -23.54562128115268|-46.63929204800168|       sao paulo|               SP|
|                       1046|-23.546081127035535|-46.64482029837157|       sao paulo|               SP|
|                       1046| -23.54612896641469|-46.64295148361138|       sao paulo|               SP|
|                       1041|  -23.5443921648681|-46.63949930627844|       sao paulo

In [48]:
customers_df = dfs["customers"]
sellers_df = dfs["sellers"]
geolocation_df = dfs["geolocation"]

# CP sans correspondance (left_anti)
customers_without_geo = customers_df.join(
    geolocation_df,
    customers_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,
    "left_anti"
)
print(f"Clients sans géolocalisation : {customers_without_geo.count()}")

sellers_without_geo = sellers_df.join(
    geolocation_df,
    sellers_df.seller_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,
    "left_anti"
)
print(f"Vendeurs sans géolocalisation : {sellers_without_geo.count()}")

# Comptage des CP uniques
unique_zip_customers = customers_df.select("customer_zip_code_prefix").distinct().count()
unique_zip_sellers = sellers_df.select("seller_zip_code_prefix").distinct().count()
unique_zip_geolocation = geolocation_df.select("geolocation_zip_code_prefix").distinct().count()

print(f"CP uniques dans customers : {unique_zip_customers}")
print(f"CP uniques dans sellers : {unique_zip_sellers}")
print(f"CP uniques dans geolocation : {unique_zip_geolocation}")

# Vérification de la duplication lors de la jointure
rows_before_join = customers_df.count()

join_result = customers_df.join(
    geolocation_df,
    customers_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix,
    "left"
)

rows_after_join = join_result.count()

print(f"Avant la jointure : {rows_before_join}")
print(f"Après la jointure : {rows_after_join}")

Clients sans géolocalisation : 278


Vendeurs sans géolocalisation : 7
CP uniques dans customers : 14994
CP uniques dans sellers : 2246
CP uniques dans geolocation : 19015


Avant la jointure : 99441
Après la jointure : 15083733


In [50]:
from pyspark.sql import functions as F

geo_clean = geolocation_df.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("lat"),
    F.avg("geolocation_lng").alias("lng"),
    F.first("geolocation_city").alias("city"),
    F.first("geolocation_state").alias("state")
)

print(f"Lignes en geolocation originel: {geolocation_df.count()}")
print(f"Lignes em geolocation agrégé: {geo_clean.count()}")

Lignes en geolocation originel: 1000163
Lignes em geolocation agrégé: 19015


In [51]:
result_clean_join = customers_df.join(
    geo_clean,
    customers_df.customer_zip_code_prefix == geo_clean.geolocation_zip_code_prefix,
    "left"
)

print(f"Avant: {customers_df.count()}")
print(f"Après (avec geo agrégé): {result_clean_join.count()}")

Avant: 99441
Après (avec geo agrégé): 99441


In [52]:
sellers_join_result = sellers_df.join(
    geo_clean,
    sellers_df.seller_zip_code_prefix == geo_clean.geolocation_zip_code_prefix,
    "left"
)

print(f"Avant : {sellers_df.count()}")
print(f"Après (avec geo agrégé) : {sellers_join_result.count()}")

Avant : 3095
Après (avec geo agrégé) : 3095
